# 손실 함수는 선택이 아니라 결론이다

1. y = f(x) + ε, ε~N(0,σ²) 라면 **MSE** 는 로그우도를 최대화하는 것과 같다(σ 고정).
2. y∈{0,1} ~ Bernoulli(p) 라면 **교차엔트로피**를 최소화하는 p̂ 는 우도를 최대화하는 p̂ 와 같다.
여기서 격자해상도 0.001 이 '완전 일치'의 뜻을  규정한다. 격자를 굵게 쓰면 0이 아니게 된다.

In [1]:
import numpy as np, json, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
import probability as P
d = json.loads(pathlib.Path('reports/probability.json').read_text(encoding='utf-8'))
print(json.dumps(d['MSE_MLE_동치'], ensure_ascii=False, indent=1))

{
 "기울기_MSE최소": 1.486,
 "기울기_로그우도최대": 1.486,
 "차이": "0.000e+00",
 "σ²_참값추정": 2.2366,
 "해석": "σ 고정 시 두 목적함수는 상수배·상수합의 관계 → 인자가 같다(정규화 상수는 기울기에 무영향)"
}


## 정규·베르누이 분포 그림

그림 파일은 `figures/06_normal_pdf.png`, `figures/07_bernoulli_pmf.png`. 넓이가 1이 되는 건 **적분 구간이 무한할 때**이고, [-8,8] 에서 σ=3 은 0.9923 으로 잘린다.

In [2]:
x = np.linspace(-8, 8, 400)
for s in (1.0, 3.0):
    print(f'σ={s}: peak={P.normal_pdf(0,0,s):.6f} ∫= {np.trapezoid(P.normal_pdf(x,0,s), x):.6f}')

σ=1.0: peak=0.398942 ∫= 1.000000
σ=3.0: peak=0.132981 ∫= 0.992338


## 교차엔트로피 = MLE, 그리고 깁스 부등식의 범위

In [3]:
print(json.dumps(d['CE_MLE_동치'], ensure_ascii=False, indent=1))

{
 "로그우도_최대의_p": 0.348,
 "BCE_최소의_p": 0.348,
 "차이": "0.000e+00",
 "표본_평균": 0.348,
 "플러그인_엔트로피": 0.6462,
 "진_엔트로피": 0.647447,
 "참값대_ fitted_교차엔트로피": 0.647455,
 "깁스_성립": true,
 "플러그인_편향": "-0.0012",
 "해석": "CE(=BCE) 최소와 MLE 는 같은 p(0.3480) 를 고른다(격자 0.001 이내 차이 0). 그리고 깁스 부등식은 **참값 기준** CE 0.6475 ≥ H 0.6474 로 성립하고, 표본 자체의 플러그인 엔트로피(0.6462)는 -0.0012 만큼 **아래로** 작다 → '모델 손실이 엔트로피보다 크면 이상하다'는 문장은 표본 기준으로는 틀린다(플러그인 편향)"
}


플러그인(표본 자체) 엔트로피는 참값 엔트로피보다 **작게** 나온다(아래 편향). 그러므로 '손실이 엔트로피보다 크면 이상하다'는 문장은 표본 기준으로는 틀린다.

In [4]:
c = d['CE_MLE_동치']
print('플러그인', c['플러그인_엔트로피'], '참값 H', c['진_엔트로피'], '편향', c['플러그인_편향'],
      '참값 대비 CE', c['참값대_ fitted_교차엔트로피'], '깁스 성립', c['깁스_성립'])

플러그인 0.6462 참값 H 0.647447 편향 -0.0012 참값 대비 CE 0.647455 깁스 성립 True


## softmax 의 수치 안정성은 '큰 스케일에서만' 검증된다

In [5]:
z = np.array([1.,2.,3.])
with np.errstate(over='ignore'):
    print('원본 차이', np.max(np.abs(P.softmax(z,False)-P.softmax(z,True))))
    print('×1000 naive', P.softmax(z*1000,False), '\n×1000 safe  ', P.softmax(z*1000,True))

원본 차이 1.1102230246251565e-16
×1000 naive [nan nan nan] 
×1000 safe   [0. 0. 1.]


/home/user/work/missions/codyssey-A2-1-mission/src/probability.py:27: RuntimeWarning: invalid value encountered in divide
  return e / e.sum()
